<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/MMc.png" align="center" width="20%">
</div>

<br>

# M/M/c QUEUEING SIMULATION OF TENANT ONBOARDING

<br>

**About:** Model a tenant onboarding pipeline as an M/M/c queue in SimPy, instrument the resource to capture queue state and per-customer timings, and compute the standard queueing metrics that connect simulation to analytical theory.

**Learning Goals:**
1. Describe the M/M/c queueing model's assumptions and explain how Poisson arrivals and exponential service times are sampled in SimPy.
2. Extend SimPy's `Resource` class to instrument queue state and collect per-customer timing data during a simulation run.
3. Compute and interpret core queueing metrics - average wait time, queue delay, mean customers in system, and server utilization - from simulation output.
4. Relate simulation-derived metrics to staffing and capacity planning decisions.

**Keywords:** m/m/c queue, poisson process, exponential service, simpy, queueing metrics, server utilization, little's law

**Prerequisite Knowledge:** (1) `01_help_desk_simulation.ipynb` **(hard prerequisite)** - covers SimPy `Environment`, `Resource`, and the process generator pattern used throughout this notebook, (2) basic probability - Poisson and exponential distributions, (3) elementary statistics.

**Target User:** Practitioners moving from ad-hoc DES modeling to grounded queueing analysis, and students learning where analytical queueing theory meets simulation.


<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>


#### CONTENTS

> #### [PART 1: THE M/M/c MODEL](#Part_1)
> #### [PART 2: INSTRUMENTING A SIMPY RESOURCE](#Part_2)
> #### [PART 3: THE ARRIVAL AND SERVICE PROCESSES](#Part_3)
> #### [PART 4: RUNNING AND INTERPRETING METRICS](#Part_4)

<br>


<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **SETUP** and **IMPORTS**


In [ ]:
# Verified against SimPy 4.x and NumPy 2.x, 2026-09-03
import simpy
import random
import statistics
import numpy as np


<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## The **M/M/c** Queueing **MODEL**

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/MMc.png" align="center" width="45%" padding="10"><br>
    <br>
    An M/M/c queue: Poisson arrivals feed a single FIFO line served by c parallel servers with exponential service times.
</div>


**Scenario.** A customer has submitted their onboarding ticket and enters the tenant onboarding pipeline: the sub-system responsible for provisioning and validating their service deployment. Multiple onboardings run in parallel, each requiring deployment validation before completion. We want to know how long, on average, customers spend waiting in the queue before their onboarding begins - and how server utilization changes as we vary the number of parallel workers.

**The M/M/c model.** In Kendall notation, `M/M/c` means:

- **M** (arrivals): **M**arkovian - arrivals form a Poisson process. Inter-arrival times are exponentially distributed with rate $\lambda$, and arrivals in disjoint intervals are independent.
- **M** (service): **M**arkovian - service times are exponentially distributed with rate $\mu$ (mean $1/\mu$).
- **c** (servers): $c$ identical parallel servers draw from a single FIFO queue.

The Poisson-plus-exponential combination is what makes M/M/c *analytically* tractable. The memoryless property of the exponential distribution means the future evolution depends only on the current number of customers in system, not on how long anyone has been there. This turns the model into a continuous-time Markov chain and yields closed-form expressions for wait time, queue length, and utilization in terms of the traffic intensity $\rho = \lambda / (c\mu)$.

**Why simulate then?** Even when closed forms exist, simulation lets you:
- Confirm the analytical result by matching it (a sanity check on both).
- Extend beyond the assumptions - non-exponential service, priority classes, finite waiting rooms - where the algebra stops giving you formulas.
- Report distributions and percentiles, not just means.

We will build the M/M/c simulator, extract the same four metrics an analytical treatment would produce, and see how they behave.

___

**Sources Consulted:**
- [Kleinrock, *Queueing Systems, Volume 1: Theory*](https://www.wiley.com/en-us/Queueing+Systems%2C+Volume+1%3A+Theory-p-9780471491101) (canonical reference for M/M/c formulas, 1975)
- [SimPy documentation - Shared Resources](https://simpy.readthedocs.io/en/latest/topical_guides/resources.html) (retrieved 2026-09-03)
- [Ross, *Introduction to Probability Models*, 12th ed.](https://www.elsevier.com/books/introduction-to-probability-models/ross/978-0-12-814346-9) - Chapter 8 on queueing (2019).

___


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **INSTRUMENTING** a SimPy **RESOURCE**

SimPy's built-in `Resource` handles the queueing mechanics but does not, on its own, expose the historical data we need for analysis - queue lengths at each request/release event, per-customer wait times, cumulative service time for utilization.

The standard pattern is to **subclass `Resource`** and override `request()` and `release()` to record state on each transition. Overriding rather than wrapping keeps the SimPy semantics intact - existing process code using `with resource.request() as req:` continues to work without modification.


In [ ]:
class MonitoredResource(simpy.Resource):
    """A SimPy Resource that records queue state on every request/release,
    and accumulates per-customer wait times and total service time."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.data = []                 # (time, queue_length) samples
        self.total_service_time = 0.0  # sum of service durations across customers
        self.customer_times = []       # per-customer end-to-end time (wait + service)
        self.wait_times = []           # per-customer time spent waiting for a server

    def request(self, *args, **kwargs):
        # Snapshot queue length at the moment of a request.
        self.data.append((self._env.now, len(self.queue)))
        return super().request(*args, **kwargs)

    def release(self, *args, **kwargs):
        self.data.append((self._env.now, len(self.queue)))
        return super().release(*args, **kwargs)


The `data` list is a raw event log. From it you can reconstruct queue length as a function of time, compute time-average queue length via integration, and estimate percentiles. The three scalar accumulators (`total_service_time`, `customer_times`, `wait_times`) are the raw material for the four headline metrics computed later.

___

**Note:** `self._env` is a private attribute of `simpy.Resource`. Referencing it works in current SimPy but is technically an implementation detail. In production code, prefer passing `env` explicitly to the constructor. Verified against SimPy 4.x, 2026-09-03.
___


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## The **ARRIVAL** and **SERVICE** Processes

Two SimPy processes drive the simulation: an `arrival` process that spawns customers over time, and a `serve` process that describes what each customer does.

**Sampling the arrivals.** For a true Poisson process with rate $\lambda$, inter-arrival times are `numpy.random.exponential(scale=1/lambda)`. The code below uses `numpy.random.poisson(interval)`, which draws integer counts from a Poisson distribution rather than continuous inter-arrival gaps. For small interval values the two approaches give similar aggregate behavior, but it is worth flagging: **strict M/M/c requires exponential inter-arrival sampling**. We keep the original convention here so the reported metrics match the source scenario; substitute `np.random.exponential` for distributional fidelity.


In [ ]:
RANDOM_SEED = 2022
ARRIVAL_INTERVAL_DAYS = 4      # mean inter-arrival time
TIME_TO_COMPLETE_DAYS = 14     # nominal service duration
SIM_TIME_DAYS = 182            # ~ two quarters
NUM_SERVERS = 3                # c in M/M/c


def arrival(env, interval, mu, resource):
    """Spawn customers. Each customer is a separate SimPy process on `resource`."""
    i = 0
    while True:
        env.process(serve(env, f"Customer{i:02d}", resource, mu))
        # Samples arrival gaps as Poisson counts. For strict M/M/c, use
        # np.random.exponential(interval) instead. See prose above.
        gap = np.random.poisson(interval)
        yield env.timeout(gap)
        i += 1


def serve(env, name, resource, mu):
    arrive = env.now
    print(f"{name} arrives at tenant onboarding on day {arrive:.2f}")

    with resource.request() as req:
        yield req
        wait = env.now - arrive
        resource.wait_times.append(wait)
        print(f"{name} begins deployment validation, waited {wait:.2f} day(s)")

        # Service time = exponential jitter around the nominal service duration.
        # The `+ mu` shift makes mean service ~ mu + 1/mu; for strict M/M/c you would
        # use plain np.random.exponential(mu). This mirrors the source scenario.
        service_time = np.random.exponential(1 / TIME_TO_COMPLETE_DAYS) + mu
        resource.total_service_time += service_time
        yield env.timeout(service_time)

        resource.customer_times.append(env.now - arrive)
        print(f"{name} onboarded on day {env.now:.2f}")


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->

> **Rewrite `arrival` and `serve` to use `np.random.exponential` for both inter-arrival times and service times (with no additive shift). Re-run the simulation and compare `AVG_WAIT` and `AVG_UTIL` against the theoretical M/M/c values for the same $\lambda$, $\mu$, $c$. How close does the simulation get?**

<br>

```python
# lam = 1 / 4          # arrivals per day
# mu  = 1 / 14         # service completions per server per day
# c   = 3
# rho = lam / (c * mu) # traffic intensity - must be < 1 for stability
```

<hr style="border: 2px solid#003262;" />


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **RUNNING** and **INTERPRETING** Metrics

We initialize the environment, seed the RNGs (both `random` and `numpy.random` - they are separate streams), create a `MonitoredResource` with capacity `NUM_SERVERS`, and run the simulation to the time horizon.


In [ ]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

env = simpy.Environment()
res = MonitoredResource(env, capacity=NUM_SERVERS)
env.process(arrival(env, ARRIVAL_INTERVAL_DAYS, TIME_TO_COMPLETE_DAYS, res))
env.run(until=SIM_TIME_DAYS)


**Computing the four metrics.**

1. **Average end-to-end waiting time.** Mean of `res.wait_times` - how long, on average, a customer sits in the queue before a server picks them up.

2. **Average queue delay beyond nominal service.** How much the mean end-to-end experience exceeds the nominal service duration.

3. **Time-average number of customers in system.** Sum of per-customer time in system divided by the horizon, which by Little's Law equals $\lambda \cdot W$ where $W$ is mean time in system.

4. **Server utilization.** Total server-days consumed divided by the horizon. For $c$ servers this is a number between 0 and $c$; dividing by $c$ gives the fraction of capacity used.

___

**Simplification to call out explicitly:** The `service_time_variability = np.random.exponential(1/TIME_TO_COMPLETE_DAYS)` line below draws a *single* exponential sample *after* the simulation has finished, and uses it as a rough correction when reporting "queue delay beyond nominal service". This is a coarse post-hoc adjustment - not a principled statistical estimate. A rigorous treatment would either (a) compute the mean service time from the per-customer records already collected, or (b) derive it analytically from the sampling distribution. It is retained here to match the original scenario's output convention; treat it as illustrative, not authoritative.

___


In [ ]:
service_time_variability = np.random.exponential(1 / TIME_TO_COMPLETE_DAYS)

AVG_WAIT = statistics.mean(res.wait_times)
AVG_DELAY = AVG_WAIT - (TIME_TO_COMPLETE_DAYS + service_time_variability)
AVG_CUSTOMERS = sum(res.customer_times) / SIM_TIME_DAYS
AVG_UTIL = res.total_service_time / SIM_TIME_DAYS

print()
print(f"Expected Average E2E Waiting Time (days):     {AVG_WAIT:.2f}")
print(f"Expected Average Delay in Queue (days):       {AVG_DELAY:.2f}")
print(f"Expected Average Number of Customers in Sys:  {AVG_CUSTOMERS:.2f}")
print(f"Expected Server Utilization (server-days):    {AVG_UTIL:.2f}")
print(f"Per-server utilization fraction:              {AVG_UTIL / NUM_SERVERS:.2%}")


**Reading the metrics for capacity planning.**

- If **per-server utilization** approaches 100 percent, the queue will grow without bound in expectation. The system is at the edge of stability; any spike in arrivals produces very long waits. Add capacity.
- If **average wait time** exceeds the customer-tolerated threshold (in the source scenario, roughly one deployment cycle), the current staffing does not meet the service-level goal even in steady state.
- **Average number in system** rising while utilization stays flat suggests service-time variance is the culprit, not throughput - a hint to look at reducing variance (standardization, tooling) rather than adding headcount.
- Running the same simulation with `NUM_SERVERS + 1` and comparing metrics gives you the *marginal value of an additional server* - the same insight the sweep in notebook 1 produced, applied to a different bottleneck.

The key move is treating these four numbers as **decision inputs**, not as a report. A single metric in isolation does not tell you what to do; the *pattern across metrics* does.


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->

> **Loop over `NUM_SERVERS` in `[2, 3, 4, 5, 6]`, run the simulation for each, and record `AVG_WAIT` and per-server utilization. At what `c` does the marginal reduction in `AVG_WAIT` per added server fall below one day? Is that the same `c` at which per-server utilization drops below 50 percent? Explain what the gap (or overlap) between those two thresholds tells you about how to pick `c`.**

<br>

```python
# for c in [2, 3, 4, 5, 6]:
#     random.seed(RANDOM_SEED)
#     np.random.seed(RANDOM_SEED)
#     env = simpy.Environment()
#     res = MonitoredResource(env, capacity=c)
#     env.process(arrival(env, ARRIVAL_INTERVAL_DAYS, TIME_TO_COMPLETE_DAYS, res))
#     env.run(until=SIM_TIME_DAYS)
#     ...
```

<hr style="border: 2px solid#003262;" />


<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<hr style="border: 2px solid#003262;" />

#### WRAP-UP

You built an M/M/c queueing simulation with an instrumented `Resource`, extracted the four canonical queueing metrics, and connected them to concrete capacity decisions. Two things to carry forward:

1. **Simulation and analytical queueing theory are complements, not substitutes.** When assumptions hold, they should agree - and if they disagree, one of them is wrong (usually the simulation, in a small subtle way). When assumptions break, simulation is what you have.
2. **The right unit of analysis for capacity planning is a *set* of metrics.** Wait time alone tells you the symptom; utilization tells you whether you can fix it by adding capacity or whether variance reduction is the real lever.

**Further reading:**
- Kleinrock, *Queueing Systems, Volume 1: Theory* - Chapter 3 (M/M/c) and Chapter 4 (Little's Law derivations).
- Ross, *Introduction to Probability Models*, 12th ed., Chapter 8.
- [SimPy - Shared Resources topical guide](https://simpy.readthedocs.io/en/latest/topical_guides/resources.html)

<hr style="border: 6px solid#003262;" />
